# Distribution


## Bath Correlation Function Decomposition

One of the most critical steps in the hierarchical equations of motion (HEOM) formalism is the decomposition of the bath correlation function. **TENSO** implements the two most widely used approaches for this purpose: the traditional Matsubara decomposition and the more efficient Padé $(N-1)/N$ scheme.


In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Spectral decomposition of the Bose-Einstein distribution function.

This module provides utilities for decomposing the Bose-Einstein distribution into a sum-over-poles representation,
as required for the efficient simulation of open quantum systems using the hierarchical equations of motion (HEOM) 
and related approaches (see Hu et al., J. Chem. Phys. 134, 244106 (2011); Chen & Franco, J. Chem. Phys. 160, 204116 (2024)).
"""

from __future__ import annotations

from typing import Literal, Optional
import numpy as np
from numpy.typing import NDArray

PI = np.pi   # Numerical value of π for compactness and readability
as_array = np.array  # Alias for concise array creation

def _tridiag_eigsh(subdiag: NDArray) -> NDArray:
    """
    Compute the eigenvalues of a symmetric tridiagonal matrix with a given subdiagonal.
    
    This mathematical step is central to the Padé decomposition: 
    the eigenvalues of specific tridiagonal matrices are analytically related to the
    optimal poles of the Padé expansion of the Bose-Einstein function (see Hu et al., Appendix A).
    """
    mat = np.diag(subdiag, -1) + np.diag(subdiag, 1)
    return np.sort(np.linalg.eigvalsh(mat))[::-1]  # Sorted in descending order

class BoseEinstein(object):
    """
    Class for the spectral decomposition of the Bose-Einstein distribution.
    Provides Padé (default) and Matsubara expansions, essential for efficient bath correlation representations.
    """
    decomposition_method = 'Pade'       # Method for decomposition ('Pade' or 'Matsubara')
    pade_type = '(N-1)/N'               # Specific Padé scheme; (N-1)/N is optimal for HEOM applications
    underflow = 1.0e-14                 # Threshold to avoid division-by-zero and numerical artifacts

    def __init__(self, n: int = 0, beta: Optional[float] = None) -> None:
        """
        Parameters
        ----------
        n : int
            Number of poles in the decomposition. Controls the accuracy and cost of the expansion.
        beta : float or None
            Inverse temperature, β = 1/kT. If None, the T=0 limit is assumed.
        """
        if beta is not None:
            assert beta >= 0, "Inverse temperature β must be non-negative."
        self.n = n
        self.beta = beta

    @property
    def zero_fluctuation(self) -> float:
        """
        Return the vacuum (zero-temperature) fluctuation contribution to the bath correlation function.
        This term appears as a high-temperature prefactor in HEOM expansions and is relevant in the β→0 limit.
        """
        return 2.0 / self.beta

    def __str__(self) -> str:
        """
        Human-readable description, summarizing the decomposition scheme and physical parameters.
        """
        if self.decomposition_method == 'Pade':
            info = f'Padé[{self.pade_type}]'
        else:
            info = self.decomposition_method
        return f'Bose-Einstein at β = {self.beta:.4f} ({info}; N={self.n})'

    def ht_function(self, w: complex) -> complex:
        """
        High-temperature (β→0) approximation to the Bose-Einstein distribution.
        
        Returns
        -------
        float
            Value of the function: 0.5 + 1/(β w)
        
        This form provides a useful reference for benchmarking and sanity checks,
        as it captures the leading-order behavior for large temperatures.
        """
        beta = self.beta
        assert beta is not None, "β (inverse temperature) must be specified."
        return 0.5 + 1.0 / (beta * w)
        # The full form would be: 1.0 / (1.0 - np.exp(-beta * w))

    def function(self, w: complex) -> complex:
        """
        Evaluate the Bose-Einstein distribution at energy w, for a given inverse temperature β.
        
        For β=None, implements the T=0 occupation: step-like behavior.
        
        Returns
        -------
        float or ndarray
            Occupation number at energy w.
        """
        beta = self.beta
        if beta is None:
            # Zero-temperature limit: occupation is 1 for positive w, 0 for negative w, 0.5 at w ≈ 0.
            assert np.allclose(np.imag(w), 0)
            ans = np.where(w > self.underflow, 1.0,
                           np.where(w < -self.underflow, 0.0, 0.5))
            return ans
        else:
            # Standard Bose-Einstein occupation:
            return 1.0 / (1.0 - np.exp(-beta * w))

    def odd(self, w: float) -> float:
        """
        Return the odd part of the Bose-Einstein distribution, sometimes useful in analytical decompositions.
        For β=None (zero temperature), returns the expected step-like function.
        """
        beta = self.beta
        if beta is None:
            return 1.0 if w > 0 else 0.0
        else:
            # Analytical odd part: related to hyperbolic tangent structure in Matsubara/Padé decompositions
            return 0.5 / np.tanh(beta * w / 2)

    def even(self, w: float) -> float:
        """
        Return the even part of the Bose-Einstein distribution (constant, equals 0.5).
        Rarely required in typical open-system calculations.
        """
        return 0.5

    @property
    def residues(self) -> list[tuple[complex, complex]]:
        """
        Compute the list of (residue, pole) pairs for the chosen decomposition.
        
        These pairs parameterize the bath correlation function as a sum of exponentials:
            C(t) = Σ residue_j * exp(-pole_j * t)
        which is the key ingredient for the numerically efficient and physically meaningful construction of HEOM
        and for the quasiparticle mapping in bexcitonics.
        
        Returns
        -------
        list of (complex, complex)
            Each entry is (residue, pole)
        """
        method = NotImplemented
        if self.decomposition_method == 'Pade':
            if self.pade_type == '(N-1)/N':
                method = self.pade1
        elif self.decomposition_method == 'Matsubara':
            method = self.matsubara

        if method is NotImplemented:
            raise NotImplementedError

        n = self.n
        b = self.beta
        if b is None or n == 0:
            ans = []
        else:
            residues, zetas = method(n)
            # Both residues and poles are rescaled by β for consistency with open-system correlation function conventions.
            # The imaginary unit is applied to poles to ensure time evolution via exp(-iωt).
            ans = [(r / b, -1.0j * z / b) for r, z in zip(residues, zetas)]
        return ans

    @staticmethod
    def matsubara(n: int) -> tuple[NDArray, NDArray]:
        """
        Compute residues and poles for the Matsubara expansion.

        Poles are located at integer multiples of 2π (the Matsubara frequencies),
        and all residues are equal to unity. This scheme is simple but converges slowly at low temperature.

        Returns
        -------
        (ndarray, ndarray)
            Arrays of residues and poles, both of length n.
        """
        zetas = [2.0 * PI * (i + 1) for i in range(n)]
        residues = [1.0] * n
        return as_array(residues), as_array(zetas)

    @staticmethod
    def pade1(n: int) -> tuple[NDArray, NDArray]:
        """
        (N-1)/N Padé decomposition of the Bose-Einstein distribution.

        Implements the algorithm of Hu et al., JCP 134, 244106 (2011) Appendix A, for optimal rational expansion.
        - Constructs tridiagonal matrices whose eigenvalues yield the squared pole locations.
        - Uses two tridiagonals: one for the denominator (Q), one for the numerator (P) of the Padé approximant.
        - The residues are determined by evaluating products over differences of the roots of P and Q.
        
        Returns
        -------
        (ndarray, ndarray)
            Arrays of residues and poles, both of length n.
        """
        assert n > 0, "Number of poles must be positive."

        # Subdiagonal for the denominator tridiagonal matrix (Q). See Eq. (A9) in Hu et al.
        subdiag_q = as_array([
            1.0 / np.sqrt((2 * i + 3) * (2 * i + 5)) for i in range(2 * n - 1)
        ])
        # Poles (zetas) are related to the inverse square roots of the eigenvalues.
        zetas = 2.0 / _tridiag_eigsh(subdiag_q)[:n]
        roots_q = np.power(zetas, 2)

        # Subdiagonal for the numerator tridiagonal matrix (P). See Eq. (A11) in Hu et al.
        subdiag_p = as_array([
            1.0 / np.sqrt((2 * i + 5) * (2 * i + 7)) for i in range(2 * n - 2)
        ])
        roots_p = np.power(2.0 / _tridiag_eigsh(subdiag_p)[:n - 1], 2)

        residues = np.zeros((n, ))
        for i in range(n):
            # Compute each residue using the formula derived in Eq. (A15) of Hu et al.
            res_i = 0.5 * n * (2 * n + 3)
            if i < n - 1:
                res_i *= (roots_p[i] - roots_q[i]) / (roots_q[n - 1] - roots_q[i])
            for j in range(n - 1):
                if j != i:
                    res_i *= ((roots_p[j] - roots_q[i]) /
                              (roots_q[j] - roots_q[i]))
            residues[i] = res_i

        return as_array(residues), as_array(zetas)
